# Lesson 2: Tools

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# 1. Defining a tool using the @tool decorator

from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

square_root.invoke({"x": 16})

4.0

In [10]:
# 2. Adding to agents
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

gemini = init_chat_model(
    model="models/gemini-3.5-flash", 
    model_provider="google_genai"
)

agent = create_agent(
    model=gemini,
    tools=[square_root],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."

)

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content[0]['text'])

The square root of 467 is approximately 21.6101827849743.


In [12]:
# 4. Training cut-off without web search
question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content[0]['text'])

My training knowledge is up to date as of January 2025. 

While I don't have real-time access to the live internet for current events beyond that point, I am always ready to help you with calculations, logic, and general knowledge up to my cutoff! How can I assist you today?


In [13]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of New York City?")

{'query': 'Who is the current mayor of New York City?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_New_York_City',
   'title': 'Mayor of New York City',
   'content': 'The current mayor is Zohran Mamdani, who was elected on November 4, 2025, and took office shortly after midnight on January 1, 2026.\n\n## History\n\n[edit]\n\nSee also: List of mayors of New York City [...] | Seal of the City of New York |\n| Flag of the mayor of New York City |\n| Incumbent Zohran Mamdani since January 1, 2026 |\n| Government of New York City |\n| Style "Style (form of address)") | His Honor "Honour (style)"); Mr. Mayor (informal) |\n| Residence | Gracie Mansion |\n| Seat | New York City Hall |\n| Term length | Four years, renewable once consecutively |\n| Constituting instrument | New York City Charter |\n| Inaugural holder | Thomas Willett | [...] Gracie Mansion has been the official residence of the mayor since Fiorello 

In [15]:
web_surfing_agent = create_agent(
    model=gemini,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of New York City?")

response = web_surfing_agent.invoke(
    {"messages": [question]}
)

In [16]:
print(response['messages'][-1].content[0]['text'])

The current mayor of New York City is **Zohran Mamdani**. He assumed office on January 1, 2026, succeeding Eric Adams, after winning the November 2025 mayoral election.
